# Efficient LLM Customization Using LoRA
Implementation and evaluation protocol | TinyLlama + Dolly-15K

Execution status: training has not been run in this workspace. No measured loss, generated response, adapter size, or improvement is claimed. Run this notebook on a CUDA GPU and retain its outputs before submitting experimental conclusions.

## 1. Business context and assumptions
A team needs a compact instruction-following assistant for summarization, extraction and general writing tasks. Full fine-tuning duplicates model weights and consumes substantial optimizer memory. LoRA offers a shared frozen backbone with small task-specific adapters.

The supplied assignment omits the model setup and baseline phases. This solution assumes TinyLlama/TinyLlama-1.1B-Chat-v1.0 and adds both phases. Dolly is a general instruction dataset, so this experiment tests general instruction adaptation rather than a specific business domain. The chat checkpoint already follows instructions; improvement is an empirical question, not guaranteed.

## 2. Runtime and reproducibility
Use a fresh Python 3.11 Colab runtime with an NVIDIA T4 (16 GB). Run the installation cell before importing libraries; restart the runtime if libraries were already imported. Downloads require internet access. The setup cell removes unused Colab packages that conflict with the pinned training stack and preserves pandas 2.2.3. Version pins define an explicit API target but have not been integration-tested here. CUDA-enabled PyTorch is supplied by Colab and its version is recorded.

In [ ]:
# Colab-only setup: remove unused packages with incompatible dependencies.
# This changes only the current runtime. Restart afterward if already imported.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'gradio', 'gradio-client', 'diffusers', 'gcsfs',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.3', 'trl==0.26.0', 'peft==0.18.0',
    'accelerate==1.12.0', 'datasets==4.4.1',
    'huggingface-hub==0.36.2', 'fsspec==2025.10.0',
    'pandas==2.2.3', 'matplotlib>=3.7,<4',
])


In [ ]:
import os, json, random, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import model_info, dataset_info
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer
from IPython.display import display
from importlib.metadata import version

assert torch.cuda.is_available(), 'Select a CUDA GPU runtime first.'
SEED = 42
set_seed(SEED)
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DATA_ID = 'databricks/databricks-dolly-15k'
# Preserve these resolved revisions to repeat this exact experiment.
MODEL_REV = model_info(MODEL_ID).sha
DATA_REV = dataset_info(DATA_ID).sha
OUT = Path('lora_results')
OUT.mkdir(exist_ok=True)
MAX_LENGTH = 512
SYSTEM = 'You are a helpful assistant. Follow the user instructions.'
manifest = {
    'model': MODEL_ID, 'model_revision': MODEL_REV,
    'dataset': DATA_ID, 'dataset_revision': DATA_REV,
    'seed': SEED, 'max_length': MAX_LENGTH,
    'gpu': torch.cuda.get_device_name(0),
    'versions': {p: version(p) for p in [
        'torch', 'transformers', 'trl', 'peft', 'datasets', 'accelerate']},
}
print(json.dumps(manifest, indent=2))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
assert tokenizer.chat_template, 'A chat template is required.'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REV, torch_dtype=torch.float16,
    attn_implementation='sdpa',
).to('cuda')

## 3. Dataset preparation
Merge optional context with the instruction using a labeled section. Preserve the response as the assistant turn. Remove duplicate user prompts before splitting, then reserve validation and test sets. Filter complete conversations exceeding 512 tokens rather than silently truncating away responses. This introduces a short-example bias; report the number removed.

The messages representation verifies the requested chat format. The trainer receives the same turns as conversational prompt/completion pairs, allowing response-only loss without requiring assistant-mask support in TinyLlama's template.

In [ ]:
def format_example(example):
    instruction = (example.get('instruction') or '').strip()
    context = (example.get('context') or '').strip()
    response = (example.get('response') or '').strip()
    user = instruction + ('\n\nContext:\n' + context if context else '')
    return {'messages': [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': response},
    ]}

assert format_example({'instruction': 'Q', 'response': 'A'})[
    'messages'][1]['content'] == 'Q'
assert format_example({'instruction': 'Q', 'context': 'C', 'response': 'A'})[
    'messages'][1]['content'] == 'Q\n\nContext:\nC'
raw = load_dataset(DATA_ID, revision=DATA_REV, split='train')
raw = raw.filter(lambda x: bool((x['instruction'] or '').strip())
                 and bool((x['response'] or '').strip()))
formatted = raw.map(format_example, remove_columns=raw.column_names)
seen, keep = set(), []
for i, row in enumerate(formatted):
    key = row['messages'][1]['content']
    if key not in seen:
        seen.add(key)
        keep.append(i)
formatted = formatted.select(keep)
count_before = len(formatted)
formatted = formatted.filter(lambda x: len(tokenizer.apply_chat_template(
    x['messages'], tokenize=True, add_generation_prompt=False
)) <= MAX_LENGTH)
print('Overlength conversations removed:', count_before - len(formatted))
assert len(formatted) >= 3400
splits = formatted.train_test_split(test_size=400, seed=SEED)
heldout = splits['test'].train_test_split(test_size=200, seed=SEED)
train_messages = splits['train'].shuffle(seed=SEED).select(range(3000))
val_messages, test_messages = heldout['train'], heldout['test']

def to_pair(row):
    return {'prompt': row['messages'][:-1],
            'completion': row['messages'][-1:]}

train_ds = train_messages.map(to_pair, remove_columns=['messages'])
val_ds = val_messages.map(to_pair, remove_columns=['messages'])
print('Train / validation / test:', len(train_ds), len(val_ds), len(test_messages))
print(json.dumps(train_messages[0], indent=2))
print(tokenizer.apply_chat_template(
    train_messages[0]['messages'], tokenize=False,
    add_generation_prompt=False))
for name, ds in [('train', train_messages), ('validation', val_messages),
                 ('test', test_messages)]:
    manifest[name + '_prompt_hashes'] = [hashlib.sha256(
        x['messages'][1]['content'].encode()).hexdigest() for x in ds]
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))

## 4. Baseline on fixed prompts
Capture baseline outputs before injecting adapters. Both models use identical chat formatting, greedy decoding and token limits. The test prompts are held out from training and validation. The first 20 shuffled test examples form a fixed qualitative panel; retain all results, including regressions.

In [ ]:
test_rows = [test_messages[i] for i in range(20)]
@torch.inference_mode()
def generate(model, messages):
    model.eval()
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to('cuda')
    ids = model.generate(
        **inputs, max_new_tokens=192, do_sample=False,
        use_cache=True, pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

baseline = [generate(base_model, r['messages'][:-1]) for r in test_rows]
(OUT / 'baseline.json').write_text(json.dumps(baseline, indent=2))

## 5. LoRA configuration and injection
Use rank r=8 and alpha=16: the ordinary LoRA scaling alpha/r is 2. Rank 8 offers a modest capacity and memory budget for an initial 3,000-example experiment. Alpha is a tunable scale, not a guarantee of improved accuracy. Dropout 0.05 regularizes the adapters.

Target q_proj, k_proj, v_proj and o_proj to adapt attention queries, keys, values and output mixing. The MLP, embeddings and language-model head remain frozen. Bias updates are disabled. With TinyLlama's grouped-query attention, the architecture-derived estimate is 2,252,800 adapter parameters, approximately 0.20% of the augmented model. The code measures the actual count. Compare rank 4/8/16 on validation data if additional budget is available; select once and evaluate the test set only afterward.

In [ ]:
targets = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
for target in targets:
    assert any(name.endswith('.' + target)
               for name, _ in base_model.named_modules()), target
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=targets, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,}; total: {total:,}; {100*trainable/total:.4f}%')
assert 0 < trainable / total < 0.01
assert all('lora_' in name for name, p in model.named_parameters()
           if p.requires_grad)
# PEFT normally promotes adapter weights; explicitly ensure FP32 trainables.
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.config.use_cache = False

## 6. Supervised fine-tuning
A microbatch of 1 with 16 accumulation steps gives an effective batch of 16 on one GPU. FP16 autocast suits the T4; BF16 is disabled. Frozen backbone weights are FP16 and trainable adapters are FP32. Non-reentrant gradient checkpointing reduces activation memory at the cost of recomputation. The 512-token cap further bounds memory.

Train for two epochs at 1e-4 with cosine decay, 20 warmup steps and gradient clipping at 1.0. Log every 10 optimizer steps, evaluate/save every 50, retain two checkpoints and restore the lowest-validation-loss checkpoint. These are operational safeguards, not evidence of production readiness. Packing is off to simplify response masking and avoid cross-example attention concerns.

In [ ]:
args = SFTConfig(
    output_dir=str(OUT / 'checkpoints'),
    num_train_epochs=2, learning_rate=1e-4,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=True, bf16=False, tf32=False,
    optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
    lr_scheduler_type='cosine', warmup_steps=20,
    logging_steps=10, logging_first_step=True,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, report_to='none',
    max_length=MAX_LENGTH, packing=False,
    completion_only_loss=True, assistant_only_loss=False,
    seed=SEED, data_seed=SEED, dataloader_num_workers=0,
)
# Already wrapped above: do not pass peft_config a second time.
trainer = SFTTrainer(
    model=model, args=args, train_dataset=train_ds,
    eval_dataset=val_ds, processing_class=tokenizer,
)
# Inspect actual labels before training: prompt masked, response supervised.
batch = next(iter(trainer.get_train_dataloader()))
labels = batch['labels'][0]
assert (labels == -100).any() and (labels != -100).any()
print('Supervised target:', tokenizer.decode(labels[labels != -100].tolist()))
del batch, labels
initial_eval = trainer.evaluate()
torch.cuda.reset_peak_memory_stats()
result = trainer.train()  # To resume: trainer.train(resume_from_checkpoint=True)
final_eval = trainer.evaluate()
trainer.save_state()
metrics = {'initial_validation': initial_eval, 'final_validation': final_eval,
           'training': result.metrics,
           'best_checkpoint': trainer.state.best_model_checkpoint,
           'peak_allocated_GiB': torch.cuda.max_memory_allocated()/2**30}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
history = pd.DataFrame(trainer.state.log_history)
history.to_csv(OUT / 'training_log.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 4))
for key, label in [('loss', 'Training loss'), ('eval_loss', 'Validation loss')]:
    rows = history.dropna(subset=[key])
    ax.plot(rows['step'], rows[key], marker='.', label=label)
ax.set(xlabel='Optimizer step', ylabel='Response-token cross-entropy',
       title='Measured training and validation loss')
ax.legend(); fig.tight_layout()
fig.savefig(OUT / 'loss_curve.png', dpi=180)
plt.show()

## 7. Persistence, reload and comparison
Save only the adapter and tokenizer. Report the adapter weight file size separately from tokenizer/config files. The expected FP32 payload for 2,252,800 parameters is about 8.59 MiB plus serialization metadata; this is an estimate, not a measured artifact size. A fresh model reload verifies that the saved adapter can be reused independently of the trainer.

In [ ]:
ADAPTER = OUT / 'adapter'
trainer.save_model(str(ADAPTER))
tokenizer.save_pretrained(ADAPTER)
weights = list(ADAPTER.glob('adapter_model*.safetensors'))
assert weights, 'Adapter weights were not saved.'
size_bytes = sum(p.stat().st_size for p in weights)
print(f'Adapter weights: {size_bytes:,} bytes ({size_bytes/2**20:.2f} MiB)')
print('Complete adapter folder MiB:',
      sum(p.stat().st_size for p in ADAPTER.rglob('*') if p.is_file())/2**20)
metrics['adapter_bytes'] = size_bytes
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
# Release trainer and all aliases before loading a new backbone.
del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REV, torch_dtype=torch.float16,
    attn_implementation='sdpa',
).to('cuda')
reloaded = PeftModel.from_pretrained(fresh_base, ADAPTER, is_trainable=False)
reloaded.config.use_cache = True
fine_tuned = [generate(reloaded, r['messages'][:-1]) for r in test_rows]
comparison = pd.DataFrame({
    'prompt': [r['messages'][1]['content'] for r in test_rows],
    'reference': [r['messages'][-1]['content'] for r in test_rows],
    'baseline': baseline, 'fine_tuned': fine_tuned,
})
comparison.to_csv(OUT / 'comparison.csv', index=False)
comparison.to_json(OUT / 'comparison.json', orient='records', indent=2)
for i, row in comparison.iterrows():
    print(f'\nExample {i+1}: {row["prompt"]}')
    display(pd.DataFrame({
        'Baseline': [row['baseline']], 'Fine-tuned': [row['fine_tuned']]
    }).style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
    print('Reference:', row['reference'])

## 8. Analysis and acceptance criteria
Loss observations must be written after execution. Compare the initial and best-checkpoint validation losses, inspect the curve for instability and report the actual optimizer step count. A falling training curve with rising validation loss suggests overfitting. Lower response-token loss alone does not establish better instruction following. Training loss includes dropout while validation loss does not, so their absolute levels need not match.

Score each of the 20 fixed comparisons blind to model identity on three dimensions, each 0-2: instruction compliance (ignored/partial/full), factual or contextual correctness (wrong/mixed/correct), and relevance (off-topic/partly relevant/direct). Record baseline and adapter totals out of 6, plus a short rationale. Have a second reviewer assess ambiguous cases. Report wins, ties and losses across the full panel; include representative improvements and regressions verbatim. Dolly references are useful guides, not the only valid answers.

If there is no improvement, report that outcome. Tune learning rate, epochs or rank using validation data and evaluate the final choice on a new untouched test panel. Do not invent outputs, cherry-pick successes or weaken the baseline. The requirement for visible improvement is not satisfied until actual comparisons support it.

## 9. Submission checklist and limitations
This document contains the complete implementation, but it is not an executed experiment. Before final submission, run all notebook cells on a T4, resolve any runtime issues, and include the printed formatting example, measured parameter counts, loss plot and commentary, actual adapter size, and full response comparison outputs. Replace this execution-status statement with the verified run details only after doing so.

Export the executed notebook to PDF with File > Print > Save as PDF, and check that wide tables and code are legible. Submit the PDF itself, not a PDF containing only a Colab link. Follow the supplied limits: A4 or Letter, at most 50 pages, under 20 MB, lowercase .pdf extension, and an allowed filename. Suggested name: Efficient_LLM_Customization_LoRA.pdf. Keep the notebook and saved adapters as reproducibility materials.

Limitations: the 512-token filter excludes long examples; 20 examples offer only a small qualitative panel; random splitting cannot eliminate semantic near-duplicates; a general dataset does not establish domain expertise. Review the Dolly dataset's CC BY-SA 3.0 license and model terms before redistribution or deployment.

## 10. References
[1] Databricks Dolly-15K dataset card: https://huggingface.co/datasets/databricks/databricks-dolly-15k
[2] TinyLlama chat model card: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0
[3] TRL 0.26.0 SFTTrainer documentation (prompt/completion loss, PEFT integration and configuration): https://huggingface.co/docs/trl/v0.26.0/en/sft_trainer
[4] Hu et al., LoRA: Low-Rank Adaptation of Large Language Models: https://arxiv.org/abs/2106.09685